# Common Sector Taxonomy

**Purpose:** Centralized sector/industry classification and mapping utilities.

**Usage:**
```python
%run "./common/common_sector_taxonomy"

# Get sector classification
sector = get_sector_from_category("Electronics")
industry = get_industry_group("Technology")

# Apply to DataFrame
df_enriched = enrich_with_taxonomy(df, "product_category")
```

**Features:**
* Sector classification mapping
* Industry grouping
* Category standardization
* Taxonomy enrichment for DataFrames

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, when, upper, trim, udf
from pyspark.sql.types import StringType
from typing import Optional, Dict

# ============================================================================
# SECTOR TAXONOMY MAPPINGS
# ============================================================================

# Product category to sector mapping
CATEGORY_TO_SECTOR = {
    # Technology & Electronics
    "electronics": "Technology",
    "computers": "Technology",
    "telephony": "Technology",
    "tablets_printing_image": "Technology",
    "computers_accessories": "Technology",
    "electronics_accessories": "Technology",
    
    # Home & Garden
    "home_appliances": "Home & Garden",
    "home_appliances_2": "Home & Garden",
    "furniture_decor": "Home & Garden",
    "bed_bath_table": "Home & Garden",
    "home_comfort_2": "Home & Garden",
    "garden_tools": "Home & Garden",
    "housewares": "Home & Garden",
    "home_construction": "Home & Garden",
    
    # Fashion & Beauty
    "fashion_bags_accessories": "Fashion",
    "fashion_shoes": "Fashion",
    "fashion_clothes": "Fashion",
    "fashion_underwear_beach": "Fashion",
    "fashion_sport": "Fashion",
    "perfumery": "Beauty",
    "health_beauty": "Beauty",
    
    # Sports & Leisure
    "sports_leisure": "Sports & Leisure",
    "toys": "Sports & Leisure",
    "cool_stuff": "Sports & Leisure",
    "musical_instruments": "Sports & Leisure",
    "party_supplies": "Sports & Leisure",
    "arts_and_craftmanship": "Sports & Leisure",
    
    # Books & Media
    "books_general_interest": "Media",
    "books_technical": "Media",
    "books_imported": "Media",
    "music": "Media",
    "cds_dvds_musicals": "Media",
    "dvds_blu_ray": "Media",
    
    # Auto & Tools
    "auto": "Automotive",
    "construction_tools_safety": "Tools & Hardware",
    "industry_commerce_and_business": "Business",
    
    # Other
    "baby": "Baby & Kids",
    "pet_shop": "Pets",
    "office_furniture": "Office",
    "stationery": "Office",
    "watches_gifts": "Gifts",
    "food": "Food & Beverage",
    "food_drink": "Food & Beverage",
}

# Sector to industry group mapping
SECTOR_TO_INDUSTRY_GROUP = {
    "Technology": "Consumer Electronics",
    "Home & Garden": "Home Goods",
    "Fashion": "Apparel & Accessories",
    "Beauty": "Personal Care",
    "Sports & Leisure": "Recreation",
    "Media": "Entertainment",
    "Automotive": "Transportation",
    "Tools & Hardware": "Industrial",
    "Business": "B2B Services",
    "Baby & Kids": "Family Products",
    "Pets": "Pet Care",
    "Office": "Office Supplies",
    "Gifts": "Gifts & Specialty",
    "Food & Beverage": "Food & Beverage",
}

In [0]:
def normalize_category(category: Optional[str]) -> str:
    """
    Normalize category string for consistent matching.
    
    Args:
        category: Raw category string
    
    Returns:
        Normalized category string
    
    Example:
        norm = normalize_category("  ELECTRONICS  ")
        # Returns: "electronics"
    """
    if category is None:
        return "unknown"
    
    return category.strip().lower().replace(" ", "_")

def get_sector_from_category(
    category: str,
    default: str = "Other"
) -> str:
    """
    Get sector classification from product category.
    
    Args:
        category: Product category
        default: Default sector if not found
    
    Returns:
        Sector name
    
    Example:
        sector = get_sector_from_category("electronics")
        # Returns: "Technology"
    """
    normalized = normalize_category(category)
    return CATEGORY_TO_SECTOR.get(normalized, default)

def get_industry_group(
    sector: str,
    default: str = "Other"
) -> str:
    """
    Get industry group from sector.
    
    Args:
        sector: Sector name
        default: Default industry group if not found
    
    Returns:
        Industry group name
    
    Example:
        industry = get_industry_group("Technology")
        # Returns: "Consumer Electronics"
    """
    return SECTOR_TO_INDUSTRY_GROUP.get(sector, default)

def get_full_taxonomy(
    category: str
) -> Dict[str, str]:
    """
    Get complete taxonomy classification.
    
    Args:
        category: Product category
    
    Returns:
        Dictionary with category, sector, and industry_group
    
    Example:
        taxonomy = get_full_taxonomy("electronics")
        # Returns: {
        #     "category": "electronics",
        #     "sector": "Technology",
        #     "industry_group": "Consumer Electronics"
        # }
    """
    normalized = normalize_category(category)
    sector = get_sector_from_category(category)
    industry_group = get_industry_group(sector)
    
    return {
        "category": normalized,
        "sector": sector,
        "industry_group": industry_group
    }

In [0]:
def enrich_with_taxonomy(
    df: DataFrame,
    category_column: str,
    sector_column: str = "sector",
    industry_column: str = "industry_group"
) -> DataFrame:
    """
    Enrich DataFrame with sector and industry classifications.
    
    Args:
        df: Input DataFrame
        category_column: Name of category column
        sector_column: Name for new sector column
        industry_column: Name for new industry group column
    
    Returns:
        DataFrame with added taxonomy columns
    
    Example:
        products_df = enrich_with_taxonomy(
            products_df,
            category_column="product_category"
        )
        # Adds 'sector' and 'industry_group' columns
    """
    # Create UDF for sector lookup
    sector_udf = udf(lambda cat: get_sector_from_category(cat), StringType())
    industry_udf = udf(lambda sec: get_industry_group(sec), StringType())
    
    # Add sector column
    df_with_sector = df.withColumn(
        sector_column,
        sector_udf(col(category_column))
    )
    
    # Add industry group column
    df_enriched = df_with_sector.withColumn(
        industry_column,
        industry_udf(col(sector_column))
    )
    
    return df_enriched

def get_sector_distribution(
    df: DataFrame,
    category_column: str
) -> DataFrame:
    """
    Get distribution of sectors in DataFrame.
    
    Args:
        df: Input DataFrame
        category_column: Name of category column
    
    Returns:
        DataFrame with sector counts
    
    Example:
        distribution = get_sector_distribution(products_df, "product_category")
        display(distribution.orderBy("count", ascending=False))
    """
    df_with_sector = enrich_with_taxonomy(df, category_column)
    
    return df_with_sector.groupBy("sector") \
        .count() \
        .orderBy("count", ascending=False)